In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pickle
import warnings
warnings.filterwarnings('ignore')

print(f"XGBoost version: {xgb.__version__}")

XGBoost version: 3.2.0


In [2]:
data_path = '../dataset/raw/creditcard.csv'
df = pd.read_csv(data_path)
print(f"Kích thước dữ liệu: {df.shape}")
print(df['Class'].value_counts()) 

Kích thước dữ liệu: (284807, 31)
Class
0    284315
1       492
Name: count, dtype: int64


In [3]:
# Tách Features (X) và Label (y)
X = df.drop(['Class'], axis=1)
y = df['Class']

# Chia tập train/test (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Khởi tạo và fit Scaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Đã chuẩn hóa dữ liệu xong!")

Đã chuẩn hóa dữ liệu xong!


In [4]:
# Tính toán tỷ lệ chênh lệch để bù đắp vào trọng số
neg_class = y_train.value_counts()[0]
pos_class = y_train.value_counts()[1]
scale_weight = neg_class / pos_class
print(f"Tỷ lệ scale_pos_weight: {scale_weight:.2f}")

# Khởi tạo mô hình XGBoost
xgb_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.1,
    scale_pos_weight=scale_weight,
    tree_method='hist',
    eval_metric='auc',
    random_state=42
)

#Training
print("Đang huấn luyện mô hình XGBoost...")
xgb_model.fit(X_train_scaled, y_train)
print("Huấn luyện hoàn tất!")

Tỷ lệ scale_pos_weight: 577.29
Đang huấn luyện mô hình XGBoost...
Huấn luyện hoàn tất!


In [5]:
# Dự đoán trên tập test
y_pred = xgb_model.predict(X_test_scaled)
y_pred_proba = xgb_model.predict_proba(X_test_scaled)[:, 1]

print("=== CONFUSION MATRIX ===")
print(confusion_matrix(y_test, y_pred))

print("\n=== CLASSIFICATION REPORT ===")
print(classification_report(y_test, y_pred))

print(f"\nROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")

=== CONFUSION MATRIX ===
[[56848    16]
 [   16    82]]

=== CLASSIFICATION REPORT ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.84      0.84      0.84        98

    accuracy                           1.00     56962
   macro avg       0.92      0.92      0.92     56962
weighted avg       1.00      1.00      1.00     56962


ROC-AUC Score: 0.9778


In [6]:
# Lưu Scaler
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
    
# Lưu Mô hình XGBoost
with open('../models/fraud_model.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)
    
print("Đã lưu scaler.pkl và fraud_model.pkl thành công vào thư mục models/!")

Đã lưu scaler.pkl và fraud_model.pkl thành công vào thư mục models/!
